# Play Media Streaming

Preview an `rgb_array` frame inline, then run the last cell to play slow and fast streaming actions in a pygame window.

In [ ]:
from IPython.display import display
from PIL import Image

from masa.envs.tabular.media_streaming import MediaStreaming

ENV_NAME = "media_streaming"
SEED = 0


In [ ]:
preview_env = MediaStreaming(render_mode="rgb_array", render_window_size=640)
obs, info = preview_env.reset(seed=SEED)
print("reset:", {"obs": obs, "info": info})
display(Image.fromarray(preview_env.render()))
preview_env.close()


In [ ]:
def play_env(seed=SEED):
    import pygame

    action_keys = {
        pygame.K_LEFT: 0,
        pygame.K_a: 0,
        pygame.K_RIGHT: 1,
        pygame.K_d: 1,
        pygame.K_SPACE: 1,
    }
    env = MediaStreaming(render_mode="human", render_window_size=640)
    obs, info = env.reset(seed=seed)
    print("Controls: Left/A slow, Right/D/Space fast, R resets, Q or Escape quits.")
    print("reset:", {"obs": obs, "info": info})

    try:
        running = True
        while running and not env.human_window_closed:
            action = None
            for event in pygame.event.get():
                if not env.handle_pygame_event(event):
                    running = False
                    break
                if event.type != pygame.KEYDOWN:
                    continue
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    running = False
                    break
                if event.key == pygame.K_r:
                    obs, info = env.reset(seed=seed)
                    print("reset:", {"obs": obs, "info": info})
                    continue
                action = action_keys.get(event.key, action)

            if action is None:
                env.render()
                continue

            obs, reward, terminated, truncated, info = env.step(action)
            print({
                "obs": obs,
                "reward": reward,
                "terminated": terminated,
                "truncated": truncated,
                "info": info,
            })
    finally:
        env.close()

play_env(seed=SEED)
